# EyeAI AMD3 — FastAPI AI Engine V1 Smoke Test

This notebook does not train the model. It loads the frozen Run 09 + horizontal-flip TTA package, creates the FastAPI application in-process, and tests `/health`, `/model-info`, and `/predict` without exposing a public Kaggle port.

## 1. Repository and runtime settings

Attach the exported model-package output and the prepared HYAMD dataset. The model package is discovered automatically when one unique package is available.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")

MODEL_PACKAGE_OVERRIDE = None
DATASET_MOUNT = Path("/kaggle/input/datasets/alihasan15/hymd-armd-dataset")
API_CONFIG = REPO_DIR / "configs/api/fastapi_v1.yaml"

print("Prepared dataset mount:", DATASET_MOUNT)

## 2. Clone and install the project

The API, predictor, and quality-check modules are installed from the current Git branch.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
print("Repository and package are ready.")

## 3. Resolve the frozen model package

The required directory contains `model.pth`, `model_config.yaml`, and `version.json`.

In [ ]:
def resolve_model_package(override=None):
    if override:
        path = Path(override)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path

    roots = [Path("/kaggle/working"), Path("/kaggle/input")]
    candidates = sorted({
        version_path.parent
        for root in roots
        if root.exists()
        for version_path in root.rglob("version.json")
        if (version_path.parent / "model.pth").is_file()
        and (version_path.parent / "model_config.yaml").is_file()
    })
    preferred = [path for path in candidates if path.name == "run09_tta_v1"]
    candidates = preferred or candidates
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError("No exported EyeAI model package was found.")
    raise RuntimeError(
        "Multiple model packages were found. Set MODEL_PACKAGE_OVERRIDE explicitly:\n"
        + "\n".join(f"- {path}" for path in candidates)
    )

MODEL_PACKAGE_DIR = resolve_model_package(MODEL_PACKAGE_OVERRIDE)
print("Model package:", MODEL_PACKAGE_DIR)
print(json.dumps(json.loads((MODEL_PACKAGE_DIR / "version.json").read_text()), indent=2))

## 4. Create the API and verify system endpoints

`TestClient` runs the real FastAPI lifecycle and loads the model once.

In [ ]:
from fastapi.testclient import TestClient
from eyeai.api.config import ApiSettings
from eyeai.api.main import create_app

settings = ApiSettings.from_yaml(
    API_CONFIG,
    model_package_override=MODEL_PACKAGE_DIR,
    device_override="cuda" if __import__("torch").cuda.is_available() else "cpu",
)
app = create_app(settings)
client_context = TestClient(app)
client = client_context.__enter__()

health = client.get("/health")
model_info = client.get("/model-info")
health.raise_for_status()
model_info.raise_for_status()

print("Health:")
print(json.dumps(health.json(), indent=2))
print("\nModel info:")
print(json.dumps(model_info.json(), indent=2))

## 5. Send one real fundus image to `/predict`

The response includes the TTA probabilities, threshold decision, latency, and initial quality warnings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

prepared_roots = [
    DATASET_MOUNT / "eyeai_prepared_binary_dataset",
    Path("/kaggle/working/eyeai_prepared_binary_dataset_retfound"),
]
prepared_roots.extend(
    path.parent
    for path in Path("/kaggle/input").rglob("dataset_summary.json")
    if (path.parent / "manifests/hyamd_val.csv").is_file()
)
DATASET_ROOT = next(
    (root for root in prepared_roots if (root / "manifests/hyamd_val.csv").is_file()),
    None,
)
if DATASET_ROOT is None:
    raise FileNotFoundError("The prepared HYAMD dataset was not found.")

validation = pd.read_csv(DATASET_ROOT / "manifests/hyamd_val.csv", dtype={"image_id": str})
row = validation.sample(n=1, random_state=42).iloc[0]
image_path = DATASET_ROOT / row["relative_image_path"]

suffix = image_path.suffix.lower()
content_type = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
    ".webp": "image/webp",
    ".tif": "image/tiff",
    ".tiff": "image/tiff",
}[suffix]

with image_path.open("rb") as handle:
    response = client.post(
        "/predict",
        files={"file": (image_path.name, handle, content_type)},
    )
response.raise_for_status()
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

with Image.open(image_path) as image:
    plt.figure(figsize=(7, 7))
    plt.imshow(image.convert("RGB"))
    plt.title(
        f"Truth={int(row['binary_label'])} | "
        f"Prediction={response.json()['label']} | "
        f"P(AMD)={response.json()['probability']:.4f}"
    )
    plt.axis("off")
    plt.show()

## 6. Close the in-process API client

This releases the application lifecycle after the smoke test.

In [ ]:
client_context.__exit__(None, None, None)
print("FastAPI smoke test completed.")